# MDAV — AIForge diffusion-forgery model (`best_diffusion.pth`)

Trains a **standalone** AI-/diffusion-inpainting detector on [Scam-AI/AIForge-Doc-v1](https://huggingface.co/datasets/Scam-AI/AIForge-Doc-v1) (receipts/forms, CC BY 4.0). It is the `diffusion` branch that DS-fuses with the DocTamper `visual` branch — a **separate** model, not a mix.

- Same `MDAVNet` architecture + preprocessing as the visual branch, so the backend loads it unchanged from `MDAV_DIFFUSION_WEIGHTS`.
- Output: `/kaggle/working/best_diffusion.pth`.

**Before running:** Accelerator = **GPU (T4)**, **Internet = ON** (for the HF download).

In [ ]:
!pip install -q jpegio opencv-python-headless six segmentation-models-pytorch==0.3.4 huggingface_hub

In [ ]:
import os, io, time, math, tempfile, threading
from glob import glob

import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import segmentation_models_pytorch as smp

# Preprocessing contract — MUST match backend/app/services/diffusion_service.py
_QUALITIES = (75, 80, 85, 90)
_MEAN, _STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# ---- download AIForge-Doc-v1 (Internet ON) ----------------------------------
from huggingface_hub import snapshot_download
AIFORGE = snapshot_download("Scam-AI/AIForge-Doc-v1", repo_type="dataset",
                            local_dir="/kaggle/working/aiforge")
TRAIN_ROOT = os.path.join(AIFORGE, "TrainingSet")
VAL_ROOT   = os.path.join(AIFORGE, "TestingSet")
print("AIForge at:", AIFORGE)

In [ ]:
# ---- dataset: images/ + masks/ (+ authentic/ as clean negatives) ------------
# Masks are 0/255; (!=0) binarizes them. Grayscale -> JPEG recompress -> RGB +
# quantised-DCT stream is the exact input the visual/diffusion branches expect.
class AIForgeDataset(Dataset):
    def __init__(self, root, img_size=512, minq=75, train=True):
        self.img_size, self.minq, self.train = img_size, minq, train
        imgs = sorted(glob(os.path.join(root, "images", "*.png")))
        self.samples = [(p, os.path.join(root, "masks", os.path.basename(p))) for p in imgs]
        self.samples += [(p, None) for p in sorted(glob(os.path.join(root, "authentic", "*.png")))]
        self._tmp = tempfile.mkdtemp(prefix="aiforge_")
        self.norm = T.Compose([T.ToTensor(), T.Normalize(_MEAN, _STD)])

    def __len__(self):
        return len(self.samples)

    def _qualities(self):
        if not self.train:
            return [self.minq]
        rng = np.random.default_rng()
        n = int(rng.integers(1, 4))
        return [int(rng.choice([q for q in _QUALITIES if q >= self.minq])) for _ in range(n)]

    def _dct(self, pil_L, qs):
        im = pil_L
        for q in qs[:-1]:                          # intermediate recompressions in-memory
            b = io.BytesIO(); im.save(b, "JPEG", quality=int(q)); b.seek(0)
            im = Image.open(b); im.load()
        path = os.path.join(self._tmp, f"{os.getpid()}_{threading.get_ident()}.jpg")
        im.save(path, "JPEG", quality=int(qs[-1]))
        im = Image.open(path); im.load()
        import jpegio
        dct = np.clip(np.abs(jpegio.read(path).coef_arrays[0]), 0, 20).astype(np.uint8)
        try:
            os.remove(path)
        except OSError:
            pass
        return im.convert("RGB"), dct

    def __getitem__(self, i):
        img_path, mask_path = self.samples[i]
        im = Image.open(img_path).convert("L")
        if mask_path is not None and os.path.exists(mask_path):
            mask = (cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) != 0).astype(np.int64)
        else:                                      # authentic -> all-clean label
            mask = np.zeros((im.size[1], im.size[0]), np.int64)

        if self.train and np.random.rand() < 0.5:
            im = im.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask[:, ::-1].copy()

        rgb, dct = self._dct(im, self._qualities())

        # 8-aligned crop keeps the DCT block grid lined up with RGB/mask pixels.
        H = min(dct.shape[0], mask.shape[0], rgb.size[1])
        W = min(dct.shape[1], mask.shape[1], rgb.size[0])
        h = min(self.img_size, H); w = min(self.img_size, W)
        if self.train and (H > h or W > w):
            top  = 8 * int(np.random.randint(0, (H - h) // 8 + 1))
            left = 8 * int(np.random.randint(0, (W - w) // 8 + 1))
        else:
            top = left = 0
        rgb  = rgb.crop((left, top, left + w, top + h))
        dct  = dct[top:top + h, left:left + w]
        mask = mask[top:top + h, left:left + w]
        return {
            "image": self.norm(rgb),
            "dct":   torch.from_numpy(np.ascontiguousarray(dct)),
            "label": torch.from_numpy(mask),
        }

In [ ]:
# ---- model / loss / metric --------------------------------------------------
class MDAVNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(21, 8)                 # quantised-DCT stream
        self.net = smp.Unet("resnet18", encoder_weights="imagenet",
                            in_channels=11, classes=2)
    def forward(self, image, dct):
        d = self.emb(dct.long()).permute(0, 3, 1, 2).contiguous()
        return self.net(torch.cat([image, d], dim=1))

class ComboLoss(nn.Module):
    def __init__(self, fg=3.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=torch.tensor([1.0, fg]))
    def forward(self, logits, target):
        ce = self.ce(logits, target)
        prob = logits.softmax(1)[:, 1]; t = (target == 1).float()
        inter = (prob * t).sum((1, 2))
        dice = 1 - (2 * inter + 1) / (prob.sum((1, 2)) + t.sum((1, 2)) + 1)
        return ce + dice.mean()

@torch.no_grad()
def prf(logits, target):
    pred = logits.argmax(1)
    tp = ((pred == 1) & (target == 1)).sum().item()
    fp = ((pred == 1) & (target == 0)).sum().item()
    fn = ((pred == 0) & (target == 1)).sum().item()
    return tp, fp, fn

In [ ]:
# ---- train (stateless warmup+cosine LR, AMP, wall-clock guard) ---------------
def lr_at(ep, step, spe, cfg):
    total = cfg["epochs"] * spe
    s = min(ep * spe + step, total - 1)
    warm = max(1, int(0.1 * total))
    if s < warm:
        return cfg["lr"] * (s + 1) / warm
    prog = (s - warm) / max(1, total - warm)
    return cfg["lr_min"] + 0.5 * (cfg["lr"] - cfg["lr_min"]) * (1 + math.cos(math.pi * prog))

def run(cfg):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(42); np.random.seed(42)

    tr = AIForgeDataset(cfg["train_root"], cfg["img"], train=True)
    va = AIForgeDataset(cfg["val_root"],   cfg["img"], train=False)
    print(f"train={len(tr)}  val={len(va)}")
    assert len(tr) > 0, "no training images — check AIForge folder layout / Internet"

    trl = DataLoader(tr, batch_size=cfg["batch"], shuffle=True,  num_workers=cfg["workers"],
                     pin_memory=False, drop_last=True)
    val = DataLoader(va, batch_size=cfg["batch"], shuffle=False, num_workers=cfg["workers"],
                     pin_memory=False)

    model = MDAVNet().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-2)
    scaler = torch.amp.GradScaler(enabled=cfg["amp"])
    crit = ComboLoss().to(dev)
    spe = len(trl); best = -1.0; t0 = time.time()

    for ep in range(cfg["epochs"]):
        model.train()
        for it, b in enumerate(trl):
            img = b["image"].to(dev); dct = b["dct"].to(dev); lbl = b["label"].to(dev)
            for g in opt.param_groups:
                g["lr"] = lr_at(ep, it, spe, cfg)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=dev, enabled=cfg["amp"]):
                loss = crit(model(img, dct), lbl)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if it % 50 == 0:
                print(f"e{ep} it{it}/{spe} loss {loss.item():.4f} lr {opt.param_groups[0]['lr']:.2e}")
            if time.time() - t0 > cfg["max_seconds"]:
                torch.save({"model": model.state_dict()}, cfg["ckpt_out"])
                print("wall-clock guard — saved & exit"); return

        model.eval(); TP = FP = FN = 0
        with torch.no_grad():
            for b in val:
                img = b["image"].to(dev); dct = b["dct"].to(dev); lbl = b["label"].to(dev)
                with torch.amp.autocast(device_type=dev, enabled=cfg["amp"]):
                    tp, fp, fn = prf(model(img, dct), lbl)
                TP += tp; FP += fp; FN += fn
        if dev == "cuda":
            torch.cuda.empty_cache()
        p = TP / (TP + FP + 1e-9); r = TP / (TP + FN + 1e-9)
        f1 = 2 * p * r / (p + r + 1e-9)
        print(f"[val e{ep}] F1 {f1:.4f}  P {p:.4f}  R {r:.4f}")

        torch.save({"model": model.state_dict()}, cfg["ckpt_out"])
        if f1 > best:
            best = f1
            torch.save({"model": model.state_dict()}, cfg["best_out"])
            print(f"  new best F1 {best:.4f} -> {cfg['best_out']}")
    print(f"done. best F1 {best:.4f}")

In [ ]:
run({
    "train_root": TRAIN_ROOT,
    "val_root":   VAL_ROOT,
    "img": 512, "batch": 16, "epochs": 20, "workers": 2,
    "lr": 2.0e-4, "lr_min": 1e-6, "amp": True,
    "max_seconds": 11 * 3600,
    "ckpt_out": "/kaggle/working/ckpt_diffusion.pth",
    "best_out": "/kaggle/working/best_diffusion.pth",   # == MDAV_DIFFUSION_WEIGHTS
})

## Deploy

Download `/kaggle/working/best_diffusion.pth` and mount it at the backend's `MDAV_DIFFUSION_WEIGHTS` (default `/app/models/best_diffusion.pth`). `diffusion_service` loads it automatically and the `diffusion` branch flips from *pending* → *active*, DS-fusing with the DocTamper `visual` branch.